# insurance-causal

**Causal inference for insurance pricing, built on Double Machine Learning.**

GLM coefficients measure correlation, not causation. When price changes are
correlated with risk quality — as they always are in insurance — the naive
GLM estimate of price elasticity is biased. This notebook shows how large
that bias is on realistic renewal data, and how Double Machine Learning (DML)
removes it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-causal/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-causal catboost doubleml polars scikit-learn

## Synthetic renewal data with a known causal structure

We generate 3,000 renewal policies where the **true causal effect of price
on renewal is θ = -0.6** (a 10% price increase causes a 6 percentage point
reduction in renewal probability).

The confounding is built in: high-risk customers receive larger price increases
*and* have lower baseline renewal rates. A naive regression will attribute some
of the lapse to price when it is actually driven by risk quality.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 3_000

TRUE_THETA = -0.6  # causal effect of price on renewal

# Confounders (rating factors)
age_band     = rng.integers(0, 5, n)      # 0=young, 4=mature
ncb_years    = rng.integers(0, 6, n)      # no-claims bonus years
vehicle_age  = rng.integers(0, 16, n)
prior_claims = rng.binomial(1, 0.12, n)   # ~12% with a prior claim

# Risk score (drives both price change and renewal)
risk = 0.3 * (4 - age_band) + 0.2 * prior_claims - 0.1 * ncb_years + rng.normal(0, 0.5, n)

# Price change: positively correlated with risk (high-risk gets larger increases)
pct_price_change = 0.04 * risk + rng.normal(0, 0.05, n)
pct_price_change = np.clip(pct_price_change, -0.3, 0.5)

# Renewal: driven by risk quality (baseline) AND price (causal)
log_odds = 1.2 - 0.25 * risk + TRUE_THETA * pct_price_change + rng.normal(0, 0.3, n)
renewal  = (rng.uniform(size=n) < 1 / (1 + np.exp(-log_odds))).astype(int)

df = pd.DataFrame({
    "renewal":          renewal,
    "pct_price_change": pct_price_change,
    "age_band":         age_band,
    "ncb_years":        ncb_years,
    "vehicle_age":      vehicle_age,
    "prior_claims":     prior_claims,
})

print(f"Renewal rate: {renewal.mean():.1%}")
print(f"Mean price change: {pct_price_change.mean():.1%}")
print(f"True causal theta: {TRUE_THETA}")
df.head()

## Naive estimate — biased by confounding

A simple logistic regression of renewal on price change gives a biased
estimate. High-risk customers (lower baseline renewal) receive larger
price increases, so the naive model overstates how much of the lapse
is caused by price.

In [ ]:
from sklearn.linear_model import LogisticRegression

X_naive = df[["pct_price_change"]]
naive_model = LogisticRegression().fit(X_naive, df["renewal"])
naive_coef = float(naive_model.coef_[0][0])

print(f"Naive (biased) estimate:  {naive_coef:.4f}")
print(f"True causal effect:       {TRUE_THETA:.4f}")
print(f"Bias: {naive_coef - TRUE_THETA:.4f}")

## DML estimate — debiased by partialling out

`CausalPricingModel` wraps DoubleML with CatBoost nuisance models.
It partial out the influence of confounders from both the outcome
and the treatment, then estimates θ from the residual correlation.

This takes ~30 seconds on Colab (5-fold cross-fitting, 3,000 observations).

In [ ]:
from insurance_causal import CausalPricingModel
from insurance_causal.treatments import PriceChangeTreatment

model = CausalPricingModel(
    outcome="renewal",
    outcome_type="binary",
    treatment=PriceChangeTreatment(column="pct_price_change"),
    confounders=["age_band", "ncb_years", "vehicle_age", "prior_claims"],
    cv_folds=5,
)

model.fit(df)
ate = model.average_treatment_effect()
print(ate)

## Confounding bias report

The bias report compares the naive estimate to the DML estimate
and gives a plain-English interpretation for the pricing committee.

In [ ]:
bias_report = model.confounding_bias_report(naive_coefficient=naive_coef)
print(bias_report[["naive_estimate", "causal_estimate", "bias_pct", "interpretation"]].to_string())

## What you should see

The DML estimate should be close to -0.6 (the true θ).
The naive estimate is typically around -0.8 to -1.0 — overstating the
price sensitivity because it is picking up the risk-quality effect too.

The bias is not small. If you use the naive estimate to set renewal prices,
you will under-price relative to what customers will actually tolerate.

## Next steps

This notebook covers the core DML estimator. The full library also provides:

- **`CausalPricingModel.cate_by_segment`** — heterogeneous treatment effects
  by age band, vehicle group, or any other segmentation variable
- **`insurance_causal.autodml`** — Automatic Debiased ML via Riesz Representers
  for continuous treatment dose-response curves
- **`insurance_causal.elasticity`** — FCA-compliant renewal pricing optimisation
  using causal forests (EconML)

**GitHub:** https://github.com/burning-cost/insurance-causal  
**PyPI:** https://pypi.org/project/insurance-causal/

The full demo notebook (`notebooks/01_insurance_causal_demo.py`) covers
50,000-policy data, sensitivity analysis for unobserved confounders, and
CATE estimation by segment.